In [ ]:
pip install librosa soundfile

## Model Rebuild with Enhanced Features

This section rebuilds the audio separation model from scratch, incorporating the following changes:

1.  **Output Layer:** Sigmoid activation for a mask, multiplied with the input spectrogram.
2.  **Chunking:** 2-second chunks with 0.25-second overlap for inference, rejoined using overlap-add.
3.  **Amplitude Normalization:** Applied before converting to spectrogram.
4.  **Learning Rate Scheduler:** `ReduceLROnPlateau` with patience 5 and factor 0.5.
5.  **Validation Split:** 0.2 of the training data.
6.  **Fixed Settings:** Sample rate 16000, n_fft 512, hop_length 128.
7.  **Learning Rate:** Fixed at 0.001.

In [ ]:
import tensorflow as tf
import tensorflow.keras.layers as L
import numpy as np
import soundfile as sf
import librosa

# Define hyperparameters
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 128
CHUNK_DURATION = 2 # seconds
OVERLAP_DURATION = 0.25 # seconds
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2
BATCH_SIZE = 4 # Example batch size - Reduced from 16 for small dataset
EPOCHS = 100 # Example epochs

# Calculate chunk and overlap lengths in samples/frames
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
OVERLAP_SAMPLES = int(SAMPLE_RATE * OVERLAP_DURATION)

# For spectrograms, calculate number of frames in a chunk
# The formula for number of frames is (audio_length - n_fft) // hop_length + 1
# Assuming padding if needed, but for fixed chunks, we need to ensure this fits
# If we process fixed-length audio chunks, the spectrogram shape will be fixed.
# Let's assume the spectrogram height (frequency bins) is N_FFT // 2 + 1.
# Width (time frames) depends on CHUNK_SAMPLES, N_FFT, HOP_LENGTH.
SPECTROGRAM_HEIGHT = N_FFT // 2 + 1 # 257
SPECTROGRAM_WIDTH = (CHUNK_SAMPLES - N_FFT) // HOP_LENGTH + 1 # 247

# U-Net compatible dimensions: Find smallest multiples of 8 (2^3 pooling layers)
# that are >= SPECTROGRAM_HEIGHT and SPECTROGRAM_WIDTH
PADDED_SPECTROGRAM_HEIGHT = (SPECTROGRAM_HEIGHT + 7) // 8 * 8 # 257 -> 264
PADDED_SPECTROGRAM_WIDTH = (SPECTROGRAM_WIDTH + 7) // 8 * 8   # 247 -> 248

H_PAD_AMOUNT = PADDED_SPECTROGRAM_HEIGHT - SPECTROGRAM_HEIGHT # 264 - 257 = 7
W_PAD_AMOUNT = PADDED_SPECTROGRAM_WIDTH - SPECTROGRAM_WIDTH   # 248 - 247 = 1

In [ ]:
# This cell is now empty as its content has been moved to the top of the notebook.

In [ ]:
pip install pydub

Now that the video files are converted to WAV format and saved in their respective directories, we can proceed with training the model using these audio files.

### Model Definition (U-Net for Spectrogram Masking)

This defines a basic U-Net architecture that takes a mixed spectrogram and outputs a mask with Sigmoid activation.

### Model Compilation and Training Configuration

Configures the model with the Adam optimizer, mean squared error loss, and includes the `ReduceLROnPlateau` callback.

In [ ]:
import tensorflow as tf

# Define model input shape here to ensure it's available
model_input_shape = (PADDED_SPECTROGRAM_HEIGHT, PADDED_SPECTROGRAM_WIDTH, 1)
model = build_unet_model(model_input_shape)

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

# The model now outputs the separated spectrogram directly, so we can use standard MSE.
model.compile(optimizer=optimizer, loss='mse')

# Learning rate scheduler
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-7 # Changed min_lr to 1e-7
)

In [ ]:
import tensorflow as tf

# EarlyStopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

In [ ]:
import os
import glob
import librosa
import numpy as np
import tensorflow as tf
import tensorflow.keras.layers as L
# Removed shutil import
import matplotlib.pyplot as plt
from pydub import AudioSegment
# Removed drive import

# Redefine hyperparameters to ensure they are in scope if this cell is run independently
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 128
CHUNK_DURATION = 2 # seconds
OVERLAP_DURATION = 0.25 # seconds
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2
BATCH_SIZE = 4 # Example batch size - Reduced from 16 for small dataset
EPOCHS = 100 # Example epochs (can be adjusted)

# Calculate chunk and overlap lengths in samples/frames
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
OVERLAP_SAMPLES = int(SAMPLE_RATE * OVERLAP_DURATION)

# For spectrograms, calculate number of frames in a chunk
SPECTROGRAM_HEIGHT = N_FFT // 2 + 1
SPECTROGRAM_WIDTH = (CHUNK_SAMPLES - N_FFT) // HOP_LENGTH + 1

# U-Net compatible dimensions: Find smallest multiples of 8 (2^3 pooling layers)
PADDED_SPECTROGRAM_HEIGHT = (SPECTROGRAM_HEIGHT + 7) // 8 * 8
PADDED_SPECTROGRAM_WIDTH = (SPECTROGRAM_WIDTH + 7) // 8 * 8

H_PAD_AMOUNT = PADDED_SPECTROGRAM_HEIGHT - SPECTROGRAM_HEIGHT
W_PAD_AMOUNT = PADDED_SPECTROGRAM_WIDTH - SPECTROGRAM_WIDTH

# --- All function definitions moved to consolidated helper cell ---

# Instantiate the model
model_input_shape = (PADDED_SPECTROGRAM_HEIGHT, PADDED_SPECTROGRAM_WIDTH, 1)
model = build_unet_model(model_input_shape)

# --- Define output directories for converted WAV files (if needed later) ---
CONVERTED_CLEAN_DIR = '/content/converted_clean_audio'
CONVERTED_NOISE_DIR = '/content/converted_noise_audio'
os.makedirs(CONVERTED_CLEAN_DIR, exist_ok=True)
os.makedirs(CONVERTED_NOISE_DIR, exist_ok=True)

# --- Load and Prepare Real Audio Data for Training ---
print('Loading and preparing real audio data...')
CLEAN_AUDIO_DIR = '' # User should provide path to clean audio directory
NOISE_AUDIO_DIR = '' # User should provide path to noisy audio directory

# Check if directories are set for training
if not CLEAN_AUDIO_DIR or not NOISE_AUDIO_DIR:
    print("Warning: CLEAN_AUDIO_DIR or NOISE_AUDIO_DIR is not set. Please set them to proceed with training.")
    # Initialize empty arrays to prevent errors later, but training won't happen
    noisy_input_data = np.array([], dtype=np.float32)
    clean_audio_data = np.array([], dtype=np.float32)
else:
    try:
        noisy_input_data, clean_audio_data = load_and_prepare_audio_data(
            CLEAN_AUDIO_DIR, NOISE_AUDIO_DIR, SAMPLE_RATE, CHUNK_SAMPLES,
            CONVERTED_CLEAN_DIR, CONVERTED_NOISE_DIR
        )
        print('Audio data prepared. Total chunks: ', len(noisy_input_data))

        # Split data for training and validation
        split_idx = int(len(noisy_input_data) * (1 - VALIDATION_SPLIT))

        train_noisy_inputs = noisy_input_data[:split_idx]
        train_clean_audios = clean_audio_data[:split_idx]
        val_noisy_inputs = noisy_input_data[split_idx:]
        val_clean_audios = clean_audio_data[split_idx:]

        train_dataset = create_tf_dataset(train_noisy_inputs, train_clean_audios, BATCH_SIZE, shuffle=True)
        val_dataset = create_tf_dataset(val_noisy_inputs, val_clean_audios, BATCH_SIZE, shuffle=False)

        print(f'Training dataset size: {tf.data.experimental.cardinality(train_dataset).numpy() * BATCH_SIZE}')
        print(f'Validation dataset size: {tf.data.experimental.cardinality(val_dataset).numpy() * BATCH_SIZE}')

    except ValueError as e:
        print(f"Error preparing audio data: {e}. Skipping dataset creation.")
        noisy_input_data = np.array([], dtype=np.float32)
        clean_audio_data = np.array([], dtype=np.float32)

# --- Initialize optimizer and compile model (moved from 521060bd) ---
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss='mse')

# --- Define Callbacks (moved from 521060bd, 7a5c4743, 2381a8d6) ---

# Removed Google Drive mount logic
drive_mounted = False # Set to False as Drive is not mounted

# Define the local directory for all checkpoints
checkpoint_dir_local = '/content/unet_checkpoints_local'
os.makedirs(checkpoint_dir_local, exist_ok=True)

# Define Google Drive path for checkpoints if drive is mounted (removed)
drive_checkpoint_path = None

# First ModelCheckpoint: saves the single best model weights (to local)
checkpoint_callback_best_filepath = os.path.join(checkpoint_dir_local, 'best_model.weights.h5')

checkpoint_callback_best = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_callback_best_filepath,
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

# Second ModelCheckpoint: saves recent model weights (to local)
checkpoint_callback_recent = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir_local, 'checkpoint_epoch_{epoch:03d}.weights.h5'),
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True, # Saves only if validation loss improves
    verbose=1
)

# Custom Callback to clean up old checkpoints
class CheckpointCleanupCallback(tf.keras.callbacks.LambdaCallback):
    def __init__(self, directory, keep_latest_n=3, exclude_filenames=None):
        super().__init__()
        self.directory = directory
        self.keep_latest_n = keep_latest_n
        self.exclude_filenames = exclude_filenames if exclude_filenames is not None else []

    def on_epoch_end(self, epoch, logs=None):
        files = glob.glob(os.path.join(self.directory, '*.weights.h5'))
        files_to_consider = []
        for f in files:
            if os.path.basename(f) not in self.exclude_filenames:
                files_to_consider.append(f)

        if len(files_to_consider) > self.keep_latest_n:
            # Sort files by modification time (most recent first)
            files_to_consider.sort(key=os.path.getmtime, reverse=True)
            for file_to_delete in files_to_consider[self.keep_latest_n:]:
                print(f"Deleting old checkpoint: {file_to_delete}")
                os.remove(file_to_delete)

cleanup_callback = CheckpointCleanupCallback(
    directory=checkpoint_dir_local,
    keep_latest_n=3,
    exclude_filenames=['best_model.weights.h5']
)

# Learning rate scheduler
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-7
)

# EarlyStopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

# Combine all callbacks
all_callbacks = [lr_scheduler, checkpoint_callback_best, checkpoint_callback_recent, cleanup_callback, early_stopping]


### Early Stopping Callback

An `EarlyStopping` callback has been added to prevent overfitting by stopping training when the validation loss no longer improves for a specified number of epochs.

### Model Training Execution

This cell combines the data loading, preprocessing, dataset creation, and model training into a single, comprehensive step. It utilizes all defined hyperparameters and callbacks, including the `ReduceLROnPlateau` scheduler and the `EarlyStopping` mechanism.

### Inference with Chunking and Overlap-Add

This section provides helper functions for splitting audio into overlapping chunks, processing them through the model, and rejoining the results using the overlap-add method.

### Model Training Execution

In [ ]:
import os
import numpy as np
import soundfile as sf
import librosa # Import librosa for loading actual audio files
import tensorflow as tf # Ensure tf is imported for TF operations
import tensorflow.keras.layers as L # Ensure L is imported for model definition
from pydub import AudioSegment # For convert_audio_to_wav
import glob # For loading audio files

# Redefine hyperparameters to ensure they are in scope
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 128
CHUNK_DURATION = 2 # seconds
OVERLAP_DURATION = 0.25 # seconds

# Calculate chunk and overlap lengths in samples/frames
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
OVERLAP_SAMPLES = int(SAMPLE_RATE * OVERLAP_DURATION)

# For spectrograms, calculate number of frames in a chunk
SPECTROGRAM_HEIGHT = N_FFT // 2 + 1
SPECTROGRAM_WIDTH = (CHUNK_SAMPLES - N_FFT) // HOP_LENGTH + 1

# U-Net compatible dimensions: Find smallest multiples of 8 (2^3 pooling layers)
PADDED_SPECTROGRAM_HEIGHT = (SPECTROGRAM_HEIGHT + 7) // 8 * 8
PADDED_SPECTROGRAM_WIDTH = (SPECTROGRAM_WIDTH + 7) // 8 * 8

H_PAD_AMOUNT = PADDED_SPECTROGRAM_HEIGHT - SPECTROGRAM_HEIGHT
W_PAD_AMOUNT = PADDED_SPECTROGRAM_WIDTH - SPECTROGRAM_WIDTH

# Helper function: RMS calculation
def _rms(audio_array):
    """Calculates the Root Mean Square (RMS) of an audio array."""
    if len(audio_array) == 0:
        return 0.0
    return np.sqrt(np.mean(np.square(audio_array)))

# Helper function: Amplitude Normalization
def normalize_amplitude(audio_array):
    """Normalizes audio amplitude to a range of -1 to 1."""
    is_tf_tensor = tf.is_tensor(audio_array)
    if (is_tf_tensor and tf.size(audio_array) == 0) or (not is_tf_tensor and len(audio_array) == 0):
        return audio_array
    if is_tf_tensor:
        max_amp = tf.reduce_max(tf.abs(audio_array))
        return tf.cond(tf.equal(max_amp, 0.0), lambda: audio_array, lambda: audio_array / max_amp)
    else:
        max_amp = np.max(np.abs(audio_array))
        if max_amp > 0:
            return audio_array / max_amp
    return audio_array

# Helper function: Audio to Spectrogram conversion
def audio_to_spectrogram(audio):
    """Converts a 1D audio array to a magnitude spectrogram."""
    if len(audio) < CHUNK_SAMPLES:
        audio = np.pad(audio, (0, CHUNK_SAMPLES - len(audio)))
    elif len(audio) > CHUNK_SAMPLES:
        audio = audio[:CHUNK_SAMPLES]
    stft = tf.signal.stft(
        audio, frame_length=N_FFT, frame_step=HOP_LENGTH, fft_length=N_FFT
    )
    magnitude_spectrogram = tf.abs(stft)
    spectrogram = tf.expand_dims(magnitude_spectrogram, axis=-1)
    spectrogram = tf.transpose(spectrogram, perm=[1, 0, 2])
    spectrogram = tf.pad(spectrogram, [[0, H_PAD_AMOUNT], [0, W_PAD_AMOUNT], [0, 0]], "CONSTANT")
    return spectrogram

# Helper function: Spectrogram to Audio conversion
def spectrogram_to_audio(spectrogram, phase):
    """Converts a magnitude spectrogram back to audio using original phase."""
    if spectrogram.shape[-1] == 1:
        spectrogram = tf.squeeze(spectrogram, axis=-1)
    spectrogram = spectrogram[:SPECTROGRAM_HEIGHT, :SPECTROGRAM_WIDTH]
    spectrogram_transposed = tf.transpose(spectrogram, perm=[1, 0])
    complex_spectrogram = tf.complex(spectrogram_transposed * tf.math.cos(phase), spectrogram_transposed * tf.math.sin(phase))
    audio = tf.signal.inverse_stft(
        complex_spectrogram, frame_length=N_FFT, frame_step=HOP_LENGTH, fft_length=N_FFT
    )
    return audio

# Helper function: build_unet_model
def build_unet_model(input_shape):
    inputs = L.Input(shape=input_shape)

    # Encoder
    conv1 = L.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    conv1 = L.Conv2D(32, (3, 3), activation='relu', padding='same')(conv1)
    pool1 = L.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = L.Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
    conv2 = L.Conv2D(64, (3, 3), activation='relu', padding='same')(conv2)
    pool2 = L.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = L.Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)
    conv3 = L.Conv2D(128, (3, 3), activation='relu', padding='same')(conv3)
    pool3 = L.MaxPooling2D(pool_size=(2, 2))(conv3)

    # Bottleneck
    conv_bottleneck = L.Conv2D(256, (3, 3), activation='relu', padding='same')(pool3)
    conv_bottleneck = L.Conv2D(256, (3, 3), activation='relu', padding='same')(conv_bottleneck)

    # Decoder
    up4 = L.UpSampling2D(size=(2, 2))(conv_bottleneck)
    up4 = L.Concatenate()([up4, conv3])
    conv4 = L.Conv2D(128, (3, 3), activation='relu', padding='same')(up4)
    conv4 = L.Conv2D(128, (3, 3), activation='relu', padding='same')(conv4)

    up5 = L.UpSampling2D(size=(2, 2))(conv4)
    up5 = L.Concatenate()([up5, conv2])
    conv5 = L.Conv2D(64, (3, 3), activation='relu', padding='same')(up5)
    conv5 = L.Conv2D(64, (3, 3), activation='relu', padding='same')(conv5)

    up6 = L.UpSampling2D(size=(2, 2))(conv5)
    up6 = L.Concatenate()([up6, conv1])
    conv6 = L.Conv2D(32, (3, 3), activation='relu', padding='same')(up6)
    conv6 = L.Conv2D(32, (3, 3), activation='relu', padding='same')(conv6)

    # Output layer: Sigmoid activation for a mask
    mask_output = L.Conv2D(1, (1, 1), activation='sigmoid', padding='same')(conv6)

    # Apply the mask to the input spectrogram to get the separated spectrogram
    separated_spectrogram_output = L.Multiply()([inputs, mask_output])

    model = tf.keras.Model(inputs=inputs, outputs=separated_spectrogram_output)
    return model

# Helper function: split_audio_into_chunks
def split_audio_into_chunks(audio, chunk_samples, overlap_samples):
    """Splits audio into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(audio):
        end = start + chunk_samples
        chunk = audio[start:end]
        if len(chunk) < chunk_samples:
            chunk = np.pad(chunk, (0, chunk_samples - len(chunk)))
        chunks.append(chunk)
        start += (chunk_samples - overlap_samples)
        if start >= len(audio) - overlap_samples:
            break
    return np.array(chunks)

# Helper function: apply_overlap_add
def apply_overlap_add(chunks, chunk_samples, overlap_samples, original_length):
    """Rejoins overlapping chunks using the overlap-add method."""
    reconstructed_audio = np.zeros(original_length)
    normalization_factor = np.zeros(original_length)
    window = np.hanning(chunk_samples)

    step_size = chunk_samples - overlap_samples

    start = 0
    for i, chunk in enumerate(chunks):
        end = start + chunk_samples
        windowed_chunk = chunk * window
        current_length = min(len(windowed_chunk), original_length - start)
        if current_length > 0:
            reconstructed_audio[start:start + current_length] += windowed_chunk[:current_length]
            normalization_factor[start:start + current_length] += window[:current_length]

        start += step_size

    normalization_factor[normalization_factor == 0] = 1e-6
    reconstructed_audio = reconstructed_audio / normalization_factor

    return reconstructed_audio

# Helper function: inference_with_chunking
def inference_with_chunking(model, full_mixed_audio):
    """Performs inference on full audio with chunking and overlap-add."""
    original_length = len(full_mixed_audio)

    # 1. Normalize amplitude of the full audio
    normalized_mixed_audio = normalize_amplitude(full_mixed_audio)

    # Calculate input_rms from the original input audio
    input_rms = _rms(normalized_mixed_audio)

    # 2. Split into chunks
    mixed_audio_chunks = split_audio_into_chunks(normalized_mixed_audio, CHUNK_SAMPLES, OVERLAP_SAMPLES)

    separated_chunks_list = []

    for chunk_idx, mixed_chunk in enumerate(mixed_audio_chunks):
        stft_mixed_chunk_original_shape = tf.signal.stft(mixed_chunk, frame_length=N_FFT, frame_step=HOP_LENGTH, fft_length=N_FFT)
        mixed_chunk_phase = tf.math.angle(stft_mixed_chunk_original_shape)

        mixed_chunk_spectrogram_padded = audio_to_spectrogram(mixed_chunk)
        mixed_chunk_spectrogram_batch = tf.expand_dims(mixed_chunk_spectrogram_padded, axis=0)

        predicted_mask_padded = model.predict(mixed_chunk_spectrogram_batch, verbose=0)
        predicted_mask = tf.squeeze(predicted_mask_padded, axis=0)

        predicted_mask_cropped = predicted_mask[:SPECTROGRAM_HEIGHT, :SPECTROGRAM_WIDTH, :]

        mixed_chunk_spectrogram_original_mag = tf.abs(stft_mixed_chunk_original_shape)
        mixed_chunk_spectrogram_original_mag = tf.expand_dims(mixed_chunk_spectrogram_original_mag, axis=-1)
        mixed_chunk_spectrogram_original_mag = tf.transpose(mixed_chunk_spectrogram_original_mag, perm=[1, 0, 2])

        masked_spectrogram = tf.squeeze(mixed_chunk_spectrogram_original_mag, axis=-1) * tf.squeeze(predicted_mask_cropped, axis=-1)

        separated_audio_chunk = spectrogram_to_audio(masked_spectrogram, mixed_chunk_phase)
        separated_chunks_list.append(separated_audio_chunk.numpy())

    separated_full_audio = apply_overlap_add(np.array(separated_chunks_list), CHUNK_SAMPLES, OVERLAP_SAMPLES, original_length)

    output_rms = _rms(separated_full_audio)

    if output_rms > 1e-6:
        scaling_factor = input_rms / output_rms
        separated_full_audio = separated_full_audio * scaling_factor
        print(f"Applied RMS normalization. Scaling factor: {scaling_factor:.4f}")
    else:
        print("Skipped RMS normalization due to zero RMS in separated audio.")

    separated_full_audio = np.clip(separated_full_audio, -1.0, 1.0)

    # Convert to int16 range for saving
    separated_full_audio = (separated_full_audio * 32767).astype(np.int16)

    return separated_full_audio

# Removed Google Drive mount logic

# Define the path to the best model weights. User should provide this path.
USER_SPECIFIED_WEIGHTS_PATH = '/content/weights.weights.h5' # Changed from '/content/unet_checkpoints_local/best_model.weights.h5'

# Instantiate the model (always create the model object first)
model_input_shape = (PADDED_SPECTROGRAM_HEIGHT, PADDED_SPECTROGRAM_WIDTH, 1)
model = build_unet_model(model_input_shape)

# Try loading weights from the user-specified path
weights_loaded = False
if os.path.exists(USER_SPECIFIED_WEIGHTS_PATH):
    print(f"Loading model weights from path: {USER_SPECIFIED_WEIGHTS_PATH}")
    try:
        model.load_weights(USER_SPECIFIED_WEIGHTS_PATH)
        print("Model weights loaded successfully.")
        weights_loaded = True
    except Exception as e:
        print(f"Error loading weights from path {USER_SPECIFIED_WEIGHTS_PATH}: {e}")
        print("Proceeding with randomly initialized weights.")
else:
    print(f"Error: Model weights not found at path: {USER_SPECIFIED_WEIGHTS_PATH}. Proceeding with randomly initialized weights.\n\tFor meaningful results, please ensure the weights file exists in a publicly accessible location or is provided by the user.")

# Sanity check: after loading weights, check model output
print("\nRunning sanity check for loaded model weights...")
dummy_input = np.ones((1, *model_input_shape), dtype=np.float32) # Batch size 1
dummy_output = model.predict(dummy_input, verbose=0)
output_min = np.min(dummy_output)
output_max = np.max(dummy_output)
print(f"Model output min value: {output_min:.4f}")
print(f"Model output max value: {output_max:.4f}")

if output_min == 0.0 and output_max == 0.0:
    print("Warning: Model output is all zeros. Weights might not have loaded correctly or model is initialized to zeros.")
else:
    print("Sanity check passed: Model output is not all zeros.")


# User-provided audio file for testing. User should upload their audio file to /content/ or provide a path.
USER_AUDIO_PATH = '/content/audio.mp4' # Changed from placeholder

# Load the user's audio file
print(f"\nLoading user's audio file: {USER_AUDIO_PATH}")
if not os.path.exists(USER_AUDIO_PATH):
    print(f"Error: User audio file not found at {USER_AUDIO_PATH}. Please upload your audio file or update the path.")
    full_mixed_audio_user = None
else:
    try:
        # Load with original sample rate first
        full_mixed_audio_user, sr_user = librosa.load(USER_AUDIO_PATH, sr=None, mono=True)
        if sr_user != SAMPLE_RATE:
            print(f"Resampling user audio from {sr_user}Hz to {SAMPLE_RATE}Hz...")
            full_mixed_audio_user = librosa.resample(y=full_mixed_audio_user, orig_sr=sr_user, target_sr=SAMPLE_RATE)
            sr_user = SAMPLE_RATE # Update sr_user to reflect the new sample rate
        print(f"User audio loaded. Duration: {len(full_mixed_audio_user)/SAMPLE_RATE:.2f} seconds.")

        # Convert to WAV if it's not already, as the original helper functions might assume WAV input or handle it better
        # This step uses the convert_audio_to_wav helper from the shared cells.
        output_converted_dir = '/content/converted_user_audio'
        os.makedirs(output_converted_dir, exist_ok=True)
        output_wav_path = os.path.join(output_converted_dir, os.path.splitext(os.path.basename(USER_AUDIO_PATH))[0] + '.wav')

        if not USER_AUDIO_PATH.lower().endswith('.wav'):
            # Load using pydub for broader format support before librosa if it's not WAV
            print(f"Converting {USER_AUDIO_PATH} to WAV for processing...")
            try:
                audio_segment = AudioSegment.from_file(USER_AUDIO_PATH)
                audio_segment = audio_segment.set_frame_rate(SAMPLE_RATE)
                audio_segment = audio_segment.set_channels(1)
                audio_segment.export(output_wav_path, format="wav")
                print(f"Converted audio saved to: {output_wav_path}")
                full_mixed_audio_user, sr_user = librosa.load(output_wav_path, sr=SAMPLE_RATE, mono=True) # Reload converted WAV
            except Exception as convert_e:
                print(f"Error converting {USER_AUDIO_PATH} to WAV: {convert_e}. Proceeding with original librosa load if possible.")
        else:
            # If it's already a WAV, just use the original path
            output_wav_path = USER_AUDIO_PATH

    except Exception as e:
        print(f"Error loading user audio file {USER_AUDIO_PATH}: {e}")
        full_mixed_audio_user = None

if full_mixed_audio_user is not None:
    print("Running inference on user's uploaded audio...")
    try:
        separated_user_audio = inference_with_chunking(model, full_mixed_audio_user)
        print("Inference completed successfully.")
        print(f"Separated user audio shape: {separated_user_audio.shape}")

        # Save the separated user audio to a file
        output_user_path = '/content/separated_audio_output.wav'
        sf.write(output_user_path, separated_user_audio, SAMPLE_RATE)
        print(f"Separated user audio saved to: {output_user_path}")

    except Exception as e:
        print(f"An error occurred during inference with user audio: {e}")
else:
    print("Skipping inference as user audio could not be loaded.")

In [ ]:
import os
import glob
import librosa
import numpy as np
import tensorflow as tf # Ensure tf is imported
# Removed shutil import
import matplotlib.pyplot as plt
from pydub import AudioSegment # Added for convert_audio_to_wav
# Removed drive import

# Redefine hyperparameters to ensure they are in scope if this cell is run independently
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 128
CHUNK_DURATION = 2 # seconds
OVERLAP_DURATION = 0.25 # seconds
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2
BATCH_SIZE = 4
EPOCHS = 1000 # Updated epochs to 1000

# Calculate chunk and overlap lengths in samples/frames
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
OVERLAP_SAMPLES = int(SAMPLE_RATE * OVERLAP_DURATION)

# For spectrograms, calculate number of frames in a chunk
SPECTROGRAM_HEIGHT = N_FFT // 2 + 1
SPECTROGRAM_WIDTH = (CHUNK_SAMPLES - N_FFT) // HOP_LENGTH + 1

# U-Net compatible dimensions: Find smallest multiples of 8 (2^3 pooling layers)
PADDED_SPECTROGRAM_HEIGHT = (SPECTROGRAM_HEIGHT + 7) // 8 * 8
PADDED_SPECTROGRAM_WIDTH = (SPECTROGRAM_WIDTH + 7) // 8 * 8

H_PAD_AMOUNT = PADDED_SPECTROGRAM_HEIGHT - SPECTROGRAM_HEIGHT
W_PAD_AMOUNT = PADDED_SPECTROGRAM_WIDTH - SPECTROGRAM_WIDTH

# Instantiate the model
model_input_shape = (PADDED_SPECTROGRAM_HEIGHT, PADDED_SPECTROGRAM_WIDTH, 1)
model = build_unet_model(model_input_shape)

# --- Define output directories for converted WAV files (if needed later) ---
CONVERTED_CLEAN_DIR = '/content/converted_clean_audio'
CONVERTED_NOISE_DIR = '/content/converted_noise_audio'
os.makedirs(CONVERTED_CLEAN_DIR, exist_ok=True)
os.makedirs(CONVERTED_NOISE_DIR, exist_ok=True)

# --- Load and Prepare Real Audio Data for Training ---
print('Loading and preparing real audio data...')
CLEAN_AUDIO_DIR = '/content/clean' # User should provide path to clean audio directory
NOISE_AUDIO_DIR = '/content/noise' # User should provide path to noisy audio directory

# Check if directories are set for training
if not CLEAN_AUDIO_DIR or not NOISE_AUDIO_DIR:
    print("Warning: CLEAN_AUDIO_DIR or NOISE_AUDIO_DIR is not set. Please set them to proceed with training.")
    # Initialize empty arrays to prevent errors later, but training won't happen
    noisy_input_data = np.array([], dtype=np.float32)
    clean_audio_data = np.array([], dtype=np.float32)
else:
    try:
        noisy_input_data, clean_audio_data = load_and_prepare_audio_data(
            CLEAN_AUDIO_DIR, NOISE_AUDIO_DIR, SAMPLE_RATE, CHUNK_SAMPLES,
            CONVERTED_CLEAN_DIR, CONVERTED_NOISE_DIR
        )
        print('Audio data prepared. Total chunks: ', len(noisy_input_data))

        # Split data for training and validation
        split_idx = int(len(noisy_input_data) * (1 - VALIDATION_SPLIT))

        train_noisy_inputs = noisy_input_data[:split_idx]
        train_clean_audios = clean_audio_data[:split_idx]
        val_noisy_inputs = noisy_input_data[split_idx:]
        val_clean_audios = clean_audio_data[split_idx:]

        train_dataset = create_tf_dataset(train_noisy_inputs, train_clean_audios, BATCH_SIZE, shuffle=True)
        val_dataset = create_tf_dataset(val_noisy_inputs, val_clean_audios, BATCH_SIZE, shuffle=False)

        print(f'Training dataset size: {tf.data.experimental.cardinality(train_dataset).numpy() * BATCH_SIZE}')
        print(f'Validation dataset size: {tf.data.experimental.cardinality(val_dataset).numpy() * BATCH_SIZE}')

    except ValueError as e:
        print(f"Error preparing audio data: {e}. Skipping dataset creation.")
        noisy_input_data = np.array([], dtype=np.float32)
        clean_audio_data = np.array([], dtype=np.float32)

# --- Initialize optimizer and compile model (moved from 521060bd) ---
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss='mse')

# --- Define Callbacks (moved from 521060bd, 7a5c4743, 2381a8d6) ---

# Removed Google Drive mount logic
drive_mounted = False # Set to False as Drive is not mounted

# Define the local directory for all checkpoints
checkpoint_dir_local = '/content/unet_checkpoints_local'
os.makedirs(checkpoint_dir_local, exist_ok=True)

# Define Google Drive path for checkpoints if drive is mounted (removed)
drive_checkpoint_path = None

# First ModelCheckpoint: saves the single best model weights (to local)
checkpoint_callback_best_filepath = os.path.join(checkpoint_dir_local, 'best_model.weights.h5')

checkpoint_callback_best = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_callback_best_filepath,
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

# Second ModelCheckpoint: saves recent model weights (to local)
checkpoint_callback_recent = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir_local, 'checkpoint_epoch_{epoch:03d}.weights.h5'),
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True, # Saves only if validation loss improves
    verbose=1
)

# Custom Callback to clean up old checkpoints
class CheckpointCleanupCallback(tf.keras.callbacks.LambdaCallback):
    def __init__(self, directory, keep_latest_n=3, exclude_filenames=None):
        super().__init__()
        self.directory = directory
        self.keep_latest_n = keep_latest_n
        self.exclude_filenames = exclude_filenames if exclude_filenames is not None else []

    def on_epoch_end(self, epoch, logs=None):
        files = glob.glob(os.path.join(self.directory, '*.weights.h5'))
        files_to_consider = []
        for f in files:
            if os.path.basename(f) not in self.exclude_filenames:
                files_to_consider.append(f)

        if len(files_to_consider) > self.keep_latest_n:
            # Sort files by modification time (most recent first)
            files_to_consider.sort(key=os.path.getmtime, reverse=True)
            for file_to_delete in files_to_consider[self.keep_latest_n:]: # pylint: disable=unused-variable
                print(f"Deleting old checkpoint: {file_to_delete}")
                os.remove(file_to_delete)

cleanup_callback = CheckpointCleanupCallback(
    directory=checkpoint_dir_local,
    keep_latest_n=3,
    exclude_filenames=['best_model.weights.h5']
)

# Learning rate scheduler
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-7
)

# EarlyStopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

# Combine all callbacks
all_callbacks = [lr_scheduler, checkpoint_callback_best, checkpoint_callback_recent, cleanup_callback, early_stopping]


**Reasoning**:
The subtask requires updating the audio directories and then re-loading and preparing the new audio data, including chunking and creating TensorFlow datasets for training and validation. This will be performed in a new code cell.



In [ ]:
print('--- Preparing new training data ---')

# 1. Update the CLEAN_AUDIO_DIR and NOISE_AUDIO_DIR variables
CLEAN_AUDIO_DIR = '/content/clean-2'
NOISE_AUDIO_DIR = '/content/noise-2'

# --- Define output directories for converted WAV files (if needed later) ---
CONVERTED_CLEAN_DIR = '/content/converted_clean_audio_2' # New dir for new data
CONVERTED_NOISE_DIR = '/content/converted_noise_audio_2' # New dir for new data
os.makedirs(CONVERTED_CLEAN_DIR, exist_ok=True)
os.makedirs(CONVERTED_NOISE_DIR, exist_ok=True)

# Check if directories are set for training
if not CLEAN_AUDIO_DIR or not NOISE_AUDIO_DIR:
    print("Warning: CLEAN_AUDIO_DIR or NOISE_AUDIO_DIR is not set. Please set them to proceed with training.")
    noisy_input_data = np.array([], dtype=np.float32)
    clean_audio_data = np.array([], dtype=np.float32)
else:
    try:
        # 3. Call the load_and_prepare_audio_data function
        noisy_input_data, clean_audio_data = load_and_prepare_audio_data(
            CLEAN_AUDIO_DIR, NOISE_AUDIO_DIR, SAMPLE_RATE, CHUNK_SAMPLES,
            CONVERTED_CLEAN_DIR, CONVERTED_NOISE_DIR
        )
        print('New audio data prepared. Total chunks: ', len(noisy_input_data))

        # 4. Calculate the split_idx for the training and validation split
        split_idx = int(len(noisy_input_data) * (1 - VALIDATION_SPLIT))

        # 5. Split noisy_input_data and clean_audio_data
        train_noisy_inputs = noisy_input_data[:split_idx]
        train_clean_audios = clean_audio_data[:split_idx]
        val_noisy_inputs = noisy_input_data[split_idx:]
        val_clean_audios = clean_audio_data[split_idx:]

        # 6. Create TensorFlow datasets named train_dataset and val_dataset
        train_dataset = create_tf_dataset(train_noisy_inputs, train_clean_audios, BATCH_SIZE, shuffle=True)
        val_dataset = create_tf_dataset(val_noisy_inputs, val_clean_audios, BATCH_SIZE, shuffle=False)

        print(f'New Training dataset size: {tf.data.experimental.cardinality(train_dataset).numpy() * BATCH_SIZE}')
        print(f'New Validation dataset size: {tf.data.experimental.cardinality(val_dataset).numpy() * BATCH_SIZE}')

    except ValueError as e:
        print(f"Error preparing new audio data: {e}. Skipping dataset creation.")
        noisy_input_data = np.array([], dtype=np.float32)
        clean_audio_data = np.array([], dtype=np.float32)


### Model Training Execution, Saving, and History Plot

This cell performs the actual training of the U-Net model using the prepared datasets and configured callbacks. After training, it saves the best model weights to Google Drive and displays a plot of the training and validation loss over epochs.

In [ ]:
# --- Start training the model ---
if 'train_dataset' in locals() and tf.data.experimental.cardinality(train_dataset).numpy() > 0:
    print('Starting model training...')
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=all_callbacks
    )

    print('Model training complete.')

    # --- Training History Plot ---
    plt.figure(figsize=(12, 6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss Over Epochs')
    plt.ylabel('Loss (MSE)')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Skipping model training: No training data available. Please ensure CLEAN_AUDIO_DIR and NOISE_AUDIO_DIR are set and contain valid audio files.")


**Reasoning**:
The previous cell `d52a6b5c` has prepared the new training and validation datasets (`train_dataset` and `val_dataset`). The current subtask requires initiating a new training session with these updated datasets and plotting the training history. Cell `94476531` already contains the logic to perform model training using the `model.fit` method with `train_dataset`, `val_dataset`, and `all_callbacks`, and then plots the training and validation loss. Therefore, executing this cell will accomplish the subtask.



In [ ]:
if 'train_dataset' in locals() and tf.data.experimental.cardinality(train_dataset).numpy() > 0:
    print('Starting model training with new data...')
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=all_callbacks
    )

    print('Model training complete.')

    # --- Training History Plot ---
    plt.figure(figsize=(12, 6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss Over Epochs')
    plt.ylabel('Loss (MSE)')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Skipping model training: No new training data available. Please ensure CLEAN_AUDIO_DIR and NOISE_AUDIO_DIR are set and contain valid audio files.")

In [ ]:
import tensorflow as tf
import numpy as np

# Redefine hyperparameters to ensure they are in scope
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 128
CHUNK_DURATION = 2 # seconds
OVERLAP_DURATION = 0.25 # seconds

# Calculate chunk and overlap lengths in samples/frames
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
OVERLAP_SAMPLES = int(SAMPLE_RATE * OVERLAP_DURATION)

# For spectrograms, calculate number of frames in a chunk
SPECTROGRAM_HEIGHT = N_FFT // 2 + 1
SPECTROGRAM_WIDTH = (CHUNK_SAMPLES - N_FFT) // HOP_LENGTH + 1

# U-Net compatible dimensions: Find smallest multiples of 8 (2^3 pooling layers)
PADDED_SPECTROGRAM_HEIGHT = (SPECTROGRAM_HEIGHT + 7) // 8 * 8
PADDED_SPECTROGRAM_WIDTH = (SPECTROGRAM_WIDTH + 7) // 8 * 8

H_PAD_AMOUNT = PADDED_SPECTROGRAM_HEIGHT - SPECTROGRAM_HEIGHT
W_PAD_AMOUNT = PADDED_SPECTROGRAM_WIDTH - SPECTROGRAM_WIDTH

# Re-instantiate the model and load best weights before saving
model_input_shape = (PADDED_SPECTROGRAM_HEIGHT, PADDED_SPECTROGRAM_WIDTH, 1)
model = build_unet_model(model_input_shape)

# Path to the best weights saved during the last training session
best_weights_path = '/content/weights.weights.h5' # Changed from '/content/unet_checkpoints_local/best_model.weights.h5'

if os.path.exists(best_weights_path):
    model.load_weights(best_weights_path)
    print(f"Model weights loaded successfully from {best_weights_path}")
else:
    print(f"Warning: Best weights not found at {best_weights_path}. Model will be saved in its current (possibly untrained) state.")

In [ ]:
final_model_save_path = '/content/final_trained_model.weights.h5'
model.save_weights(final_model_save_path)
print(f"Model weights saved to: {final_model_save_path}")